# Synthetic Hierarchical Time Series Data Generation

Generates tables for testing **hierarchical reconciliation** with the MMF pipeline.

Produces leaf-level series (country → region → store) with separate hierarchy columns, then calls `run_aggregation()` to build all hierarchy levels and save the summing matrix and level tags.

**Output tables:**
- `{catalog}.{schema}.{use_case}_train_data` — all hierarchy levels (`unique_id` encodes level, e.g. `USA/California/Store1`)
- `{catalog}.{schema}.{use_case}_hierarchy_S` — summing matrix
- `{catalog}.{schema}.{use_case}_hierarchy_tags` — level tags
- `{catalog}.{schema}.{use_case}_evaluation_output` — synthetic backtest residuals (for MinTrace standalone testing)

## 1. Parameters

In [ ]:
try:
    catalog = dbutils.widgets.get("catalog")
except Exception:
    catalog = "catalog_timeseries"

try:
    schema = dbutils.widgets.get("schema")
except Exception:
    schema = "synthetic"

try:
    use_case = dbutils.widgets.get("use_case")
except Exception:
    use_case = "mmf_e2e"

try:
    n_months = int(dbutils.widgets.get("n_months"))
except Exception:
    n_months = 36

try:
    n_stores = int(dbutils.widgets.get("n_stores"))
except Exception:
    n_stores = 20

train_table = f"{catalog}.{schema}.{use_case}_train_data"
print(f"Train table:    {train_table}")
print(f"Hierarchy S:    {catalog}.{schema}.{use_case}_hierarchy_S")
print(f"Hierarchy tags: {catalog}.{schema}.{use_case}_hierarchy_tags")
print(f"Months:         {n_months}")
print(f"Stores (leaf):  {n_stores}")

## 2. Setup

In [ ]:
import subprocess, sys

_current_user = spark.sql("SELECT current_user() AS u").collect()[0]["u"]
_workspace_path = f"/Workspace/Repos/{_current_user}/many-model-forecasting"
subprocess.check_call([sys.executable, "-m", "pip", "install", f"{_workspace_path}[hierarchical]", "--quiet"])
dbutils.library.restartPython()

In [ ]:
import sys
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

_current_user = spark.sql("SELECT current_user() AS u").collect()[0]["u"]
_workspace_path = f"/Workspace/Repos/{_current_user}/many-model-forecasting"
sys.path.insert(0, _workspace_path)
from mmf_sa import run_aggregation

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

## 3. Generate Leaf-Level Series

In [ ]:
np.random.seed(42)

# Fixed regions — stores distributed round-robin
REGIONS = [
    ("USA",    "California"),
    ("USA",    "NewYork"),
    ("Europe", "France"),
    ("Europe", "Germany"),
]

leaves = []
for i in range(n_stores):
    country, region = REGIONS[i % len(REGIONS)]
    store = f"Store{i + 1}"
    leaves.append({"country": country, "region": region, "store": store})

# Random base, slope, amplitude, noise per store
bases  = np.random.uniform(100, 300, n_stores)
slopes = np.random.uniform(0.5, 2.0, n_stores)
amps   = np.random.uniform(10,  30,  n_stores)
noises = np.random.uniform(3,   10,  n_stores)

end_date = pd.Timestamp.today().replace(day=1)
dates = pd.date_range(end=end_date, periods=n_months, freq="MS")

rows = []
for i, leaf in enumerate(leaves):
    for t, ds in enumerate(dates):
        y = (
            bases[i]
            + slopes[i] * t
            + amps[i] * np.sin(2 * np.pi * t / 12)
            + np.random.normal(0, noises[i])
        )
        rows.append({
            "unique_id": leaf["store"],
            "ds":        ds.date(),
            "y":         max(0.0, round(y, 4)),
            "country":   leaf["country"],
            "region":    leaf["region"],
            "store":     leaf["store"],
        })

leaf_pdf = pd.DataFrame(rows)
leaf_sdf = spark.createDataFrame(leaf_pdf)

print(f"Leaf series generated: {leaf_pdf['unique_id'].nunique()} stores × {n_months} months = {len(leaf_pdf):,} rows")
print(f"Date range: {leaf_pdf['ds'].min()} → {leaf_pdf['ds'].max()}")
display(leaf_pdf.groupby("unique_id")[["y"]].agg(["mean","min","max"]).round(1))

## 4. Aggregate and Save All Hierarchy Levels

In [ ]:
# Write leaf data to Delta first (run_aggregation reads from Delta)
(
    leaf_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(train_table)
)
print(f"Leaf data written to: {train_table}")

# run_aggregation() reads leaf data, builds all hierarchy levels,
# overwrites train_table with all levels, and saves _hierarchy_S and _hierarchy_tags
run_aggregation(
    spark=spark,
    train_table=train_table,
    hierarchy_cols=["country", "region", "store"],
    hierarchy_s_table=f"{catalog}.{schema}.{use_case}_hierarchy_S",
    hierarchy_tags_table=f"{catalog}.{schema}.{use_case}_hierarchy_tags",
)
print("✓ Aggregation complete — all hierarchy levels written")

## 5. Verify

In [ ]:
print("=== Train table (all hierarchy levels) ===")
display(spark.sql(f"""
    SELECT
        COUNT(*)                          AS total_rows,
        COUNT(DISTINCT unique_id)         AS n_series,
        MIN(ds)                           AS min_date,
        MAX(ds)                           AS max_date,
        ROUND(AVG(y), 2)                  AS avg_y
    FROM {train_table}
""").toPandas())

print("\nSeries by hierarchy level (classified by number of '/' in unique_id):")
display(spark.sql(f"""
    SELECT
        CASE SIZE(SPLIT(unique_id, '/')) - 1
            WHEN 0 THEN 'country (level 1)'
            WHEN 1 THEN 'country/region (level 2)'
            WHEN 2 THEN 'country/region/store — leaf (level 3)'
            ELSE CONCAT(CAST(SIZE(SPLIT(unique_id, '/')) - 1 AS STRING), ' slashes')
        END AS level,
        COUNT(DISTINCT unique_id) AS n_series
    FROM {train_table}
    GROUP BY 1 ORDER BY 1
""").toPandas())

print("\n=== Hierarchy S matrix (first 5 rows) ===")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.{use_case}_hierarchy_S LIMIT 5").toPandas())

print("\n=== Hierarchy tags ===")
display(spark.sql(f"SELECT * FROM {catalog}.{schema}.{use_case}_hierarchy_tags").toPandas())

## 6. Generate Synthetic Evaluation Output (for Skill 6 standalone)

Generates `_evaluation_output` with synthetic backtest residuals — required by MinTrace to estimate forecast error covariances across hierarchy levels. Not needed for BottomUp / TopDown / MiddleOut.

In [ ]:
# Load aggregated train_data to get all hierarchy unique_ids and dates
train_pdf = spark.table(train_table).toPandas()
train_pdf["ds"] = pd.to_datetime(train_pdf["ds"])

PREDICTION_LENGTH = 3
MODEL_NAME = "StatsForecastAutoArima"
eval_rows = []

for uid, series in train_pdf.groupby("unique_id"):
    series = series.sort_values("ds").reset_index(drop=True)
    n = len(series)
    # Two backtest windows — last 8 and last 5 months
    for offset in [-8, -5]:
        start_idx = n + offset
        end_idx = start_idx + PREDICTION_LENGTH
        if start_idx < 0 or end_idx > n:
            continue
        window = series.iloc[start_idx:end_idx]
        actual = window["y"].tolist()
        # Simulate forecast with ~5% noise
        forecast = [max(0.0, round(v + np.random.normal(0, abs(v) * 0.05), 4)) for v in actual]
        eval_rows.append({
            "unique_id": uid,
            "backtest_window_start_date": window["ds"].iloc[0],
            "forecast": forecast,
            "actual": actual,
            "model": MODEL_NAME,
        })

eval_pdf = pd.DataFrame(eval_rows)
eval_table = f"{catalog}.{schema}.{use_case}_evaluation_output"

spark.createDataFrame(eval_pdf).write \
    .format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(eval_table)

print(f"✓ Evaluation output written to: {eval_table}")
print(f"  {len(eval_pdf)} backtest rows across {eval_pdf['unique_id'].nunique()} series")
display(eval_pdf.head(5))